In [5]:
README_PROMPT = """你是一位专业的技术文档专家。请为提供的 Python 模块生成一份完整、专业的 README.md 文档。

内容应包括：

1. 模块标题及一句话简介
2. 功能特性列表（使用项目符号）
3. 安装说明
4. 快速开始（包含可直接运行的代码示例）
5. API 参考（为每个公开的函数/类提供参数、返回值和使用示例）
6. 配置说明（如有环境变量，请说明其用途）
7. 错误处理说明

请使用清晰、易于开发者阅读的 Markdown 格式进行编写，内容应具体、准确，避免空泛描述。"""

In [6]:
DOCSTRING_PROMPT = """请为所有缺少文档字符串（docstring）的函数和类添加完整的 Google 风格文档字符串。

格式如下：

```python
def function(param: type) -> return_type:
    \"\"\"一句话概述。

    Args:
        param: 参数说明。

    Returns:
        返回值说明。

    Raises:
        ErrorType: 在何种情况下会抛出该异常。

    Example:
        >>> function(value)
        expected_output
    \"\"\"
    请返回添加好文档字符串后的完整 Python 文件。"""

In [7]:
from dotenv import load_dotenv

load_dotenv()

import os

model = os.getenv('MODEL_NAME', '')

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=model, temperature=0.3, extra_body={'enable_thinking': False})

In [8]:
import ast

def extract_structure(code: str) -> str:
    """Extract functions, classes, and their signatures from Python code."""
    try:
        tree = ast.parse(code)
        structure = []

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                args = [a.arg for a in node.args.args]
                structure.append(f"def {node.name}({', '.join(args)})")
            elif isinstance(node, ast.ClassDef):
                structure.append(f"class {node.name}:")
                for item in node.body:
                    if isinstance(item, ast.FunctionDef):
                        args = [a.arg for a in item.args.args]
                        structure.append(f"  def {item.name}({', '.join(args)})")

        return "\n".join(structure)
    except Exception:
        return "Could not parse structure"


In [9]:
from langchain_core.messages import SystemMessage, HumanMessage

def generate_readme(code: str):
    structure = extract_structure(code)
    message = [
        SystemMessage(content=README_PROMPT),
        HumanMessage(content=f"""代码结构：\n{structure}\n\n完整代码：\n```python\n{code}\n```""")
    ]
    return llm.stream(input=message)

In [10]:
def add_docstrings(code: str):
    message = [
        SystemMessage(content=DOCSTRING_PROMPT),
        HumanMessage(content=f"""为以下这份代码添加docstrings：\n\n```python\n{code}\n```""")
    ]
    return llm.stream(input=message)

In [11]:
SAMPLE_CODE = r'''
import hashlib
import re
from datetime import datetime


def validate_email(email: str) -> bool:
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))


def hash_password(password: str, salt: str = "") -> str:
    combined = f"{password}{salt}"
    return hashlib.sha256(combined.encode()).hexdigest()


def parse_date(date_string: str) -> datetime:
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%Y%m%d"]
    for fmt in formats:
        try:
            return datetime.strptime(date_string, fmt)
        except ValueError:
            continue
    raise ValueError(f"Unable to parse date: {date_string}")


class UserValidator:
    MIN_PASSWORD_LENGTH = 8

    def validate_username(self, username: str) -> tuple[bool, str]:
        if len(username) < 3:
            return False, "Username too short"
        if not re.match(r'^[a-zA-Z0-9_]+$', username):
            return False, "Only letters, numbers, underscores allowed"
        return True, ""

    def validate_password(self, password: str) -> tuple[bool, str]:
        if len(password) < self.MIN_PASSWORD_LENGTH:
            return False, f"Password must be at least {self.MIN_PASSWORD_LENGTH} characters"
        if not re.search(r'[A-Z]', password):
            return False, "Password must contain at least one uppercase letter"
        if not re.search(r'\d', password):
            return False, "Password must contain at least one digit"
        return True, ""
'''

In [15]:
for chunk in generate_readme(code=SAMPLE_CODE):
    print(chunk.content, end='', flush=True)

# PyUtils: 轻量级 Python 数据验证与处理工具库

**PyUtils** 是一个专注于用户输入验证、密码安全哈希及日期格式解析的轻量级 Python 工具模块。它提供了简洁、高效的 API，帮助开发者快速实现常见的数据清洗和安全校验逻辑。

## ✨ 功能特性

- **📧 邮箱验证**：基于正则表达式的高效邮箱格式校验。
- **🔐 密码哈希**：支持自定义盐值（Salt）的 SHA-256 密码哈希生成。
- **📅 智能日期解析**：自动识别并解析多种常见日期格式（如 `YYYY-MM-DD`, `DD/MM/YYYY` 等）。
- **👤 用户数据校验器 (`UserValidator`)**：
  - 用户名合法性检查（长度限制、字符集限制）。
  - 强密码策略验证（最小长度、大小写混合、数字要求）。
- **🛡️ 类型提示支持**：所有公开函数和类均包含完整的类型注解，提升 IDE 开发体验。

## 📦 安装说明

本模块为纯 Python 标准库实现，无需安装额外的第三方依赖包。您可以直接将代码复制到您的项目中，或通过以下方式集成：

1. **直接复制**：将 `pyutils.py` 文件放入您的项目目录中。
2. **导入使用**：
   ```python
   from pyutils import validate_email, hash_password, parse_date, UserValidator
   ```

> **注意**：确保您的 Python 版本 >= 3.9（以支持 `tuple[bool, str]` 类型语法）。如果使用的是 Python 3.8 或更低版本，请将返回类型改为 `Tuple[bool, str]` 并从 `typing` 模块导入。

## 🚀 快速开始

以下示例展示了如何使用 PyUtils 进行基本的用户注册数据验证：

```python
from pyutils import validate_email, hash_password, parse_date, UserValidator

# 初始化校验器
validator = UserValidator()

# 1. 验证邮箱
email = "user@example.com"
is_valid_ema

In [16]:
for chunk in add_docstrings(code=SAMPLE_CODE):
    print(chunk.content, end='', flush=True)

```python
import hashlib
import re
from datetime import datetime


def validate_email(email: str) -> bool:
    """验证电子邮件地址的格式是否有效。

    Args:
        email: 需要验证的电子邮件地址字符串。

    Returns:
        如果电子邮件地址格式有效则返回 True，否则返回 False。

    Example:
        >>> validate_email("user@example.com")
        True
        >>> validate_email("invalid-email")
        False
    """
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))


def hash_password(password: str, salt: str = "") -> str:
    """使用 SHA-256 算法对密码进行哈希处理。

    Args:
        password: 需要哈希的原始密码字符串。
        salt: 可选的盐值字符串，用于增加哈希的安全性。默认为空字符串。

    Returns:
        密码与盐值组合后的 SHA-256 哈希值的十六进制字符串表示。

    Example:
        >>> hash_password("mypassword", "salt123")
        'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'
    """
    combined = f"{password}{salt}"
    return hashlib.sha256(combined.encode()).hexdigest()


def parse_date(date_string: str) -> datetime:
    "